# 0. Import necessary libraies

In [2]:
import open3d as o3d
import numpy as np
import json
import os
from pathlib import Path
from glob import glob
from PIL import Image
import cv2
import networkx as nx

# 1. Load the full mesh

In [3]:
scene_dir = 'ScanNet_Data/data/56a0ec536c'
mesh_file = scene_dir + '/scans/mesh_aligned_0.05_semantic.ply'
segments_file = scene_dir + '/scans/segments.json'
anno_file = scene_dir + '/scans/segments_anno.json'

In [4]:
mesh = o3d.io.read_triangle_mesh(mesh_file)
vertices = np.asarray(mesh.vertices)
triangles = np.asarray(mesh.triangles)
segment_ids = np.array(json.load(open(segments_file))['segIndices'])
anno_data = json.load(open(anno_file))['segGroups']

# 2. Extract individual objects

Now we are using the ground truth segmentation masks. But in the future, shall we use other segmentation techniques rather than ground truth for generalization?

In [5]:
segment_to_instance = {}
instance_labels = {}
for group in anno_data:
    instance_id = group['objectId']
    label = group['label']
    segments_in_group = group['segments']
    for seg_id in segments_in_group:
        segment_to_instance[seg_id] = instance_id
    instance_labels[instance_id] = label

In [6]:
unique_instances = set(segment_to_instance.values())
print(f"There are {len(unique_instances)} objects.")

There are 138 objects.


- Load the full `mech`, `vertices` and `triangles`.
- Read `segments.json` to save the segmend_id of each vertex.
- Read `segments_anno.json` to construct `segment_to_instance`

Vertex -> Segment -> Instance -> Label

For each triangle which consists of 3 vertices, check the corresponding segments of the vertices. If all segments are mapped into current instance, then take this triangle as the face of current instance.

In [6]:
output_dir = scene_dir + '/extracted_individual_objects'
os.makedirs(output_dir, exist_ok=True)

for instance_id in unique_instances:
    instance_face_indices = []
    for i, tri in enumerate(triangles):
        # Check if all 3 vertices belong to the same instance
        segs = [segment_ids[v_idx] for v_idx in tri]
        if all(segment_to_instance.get(seg) == instance_id for seg in segs):
            instance_face_indices.append(i)
    
    if not instance_face_indices:
        continue
    
    sub_triangles_global = triangles[instance_face_indices]
    unique_vertex_indices, inverse_indices = np.unique(sub_triangles_global.flatten(), return_inverse=True)
    sub_vertices = vertices[unique_vertex_indices]
    sub_triangles_local = inverse_indices.reshape(sub_triangles_global.shape)
    
    submesh = o3d.geometry.TriangleMesh()
    submesh.vertices = o3d.utility.Vector3dVector(sub_vertices)
    submesh.triangles = o3d.utility.Vector3iVector(sub_triangles_local)
    submesh.compute_vertex_normals()
    
    label = instance_labels.get(instance_id, "unknown")
    filename = os.path.join(output_dir, f"object_{instance_id}_{label}.ply")
    o3d.io.write_triangle_mesh(filename, submesh)
    print(f"Saved {filename}")

Saved ScanNet_Data/data/56a0ec536c/extracted_individual_objects/object_1_whiteboard.ply
Saved ScanNet_Data/data/56a0ec536c/extracted_individual_objects/object_2_blinds.ply
Saved ScanNet_Data/data/56a0ec536c/extracted_individual_objects/object_3_wall.ply
Saved ScanNet_Data/data/56a0ec536c/extracted_individual_objects/object_4_ceiling.ply
Saved ScanNet_Data/data/56a0ec536c/extracted_individual_objects/object_5_ceiling lamp.ply
Saved ScanNet_Data/data/56a0ec536c/extracted_individual_objects/object_6_ceiling lamp.ply
Saved ScanNet_Data/data/56a0ec536c/extracted_individual_objects/object_7_ceiling lamp.ply
Saved ScanNet_Data/data/56a0ec536c/extracted_individual_objects/object_8_ceiling lamp.ply
Saved ScanNet_Data/data/56a0ec536c/extracted_individual_objects/object_9_wall.ply
Saved ScanNet_Data/data/56a0ec536c/extracted_individual_objects/object_10_ceiling lamp.ply
Saved ScanNet_Data/data/56a0ec536c/extracted_individual_objects/object_11_whiteboard.ply
Saved ScanNet_Data/data/56a0ec536c/extr

# 3. Map the extracted individual objects into Multiview Images

$X_{camera} = R \cdot X_{world} + t$, where $R$ is Rotation Matrix and $t$ is Translation vector.

$X_{image} = K \cdot X_{camera}$, where $K$ is the camera intrinsics.

In [7]:
def read_cameras_text(path):
    cameras = {}
    with open(path, 'r') as f:
        for line in f:
            if line.startswith('#') or line.strip() == '':
                continue
            elems = line.strip().split()
            camera_id, model, width, height, fx, fy, cx, cy = elems[:8]
            cameras[int(camera_id)] = {
                'fx': float(fx), 'fy': float(fy), 'cx': float(cx), 'cy': float(cy),
                'width': int(width), 'height': int(height)
            }
    return cameras

def read_images_text(path):
    images = {}
    with open(path, 'r') as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line.startswith("#") or line == "":
            i += 1
            continue
        elems = line.split()
        if len(elems) < 10:
            i += 1
            continue  # Skip invalid lines
        try:
            image_id = int(elems[0])
            qvec = np.array(list(map(float, elems[1:5])))
            tvec = np.array(list(map(float, elems[5:8])))
            camera_id = int(elems[8])
            image_name = elems[9]
            images[image_name] = {'qvec': qvec, 'tvec': tvec, 'camera_id': camera_id}
            i += 2  # Skip the following line (2D points) in COLMAP text format
        except ValueError:
            i += 1
            continue
    return images

def qvec2rotmat(qvec):
    w, x, y, z = qvec
    return np.array([
        [1-2*y**2-2*z**2, 2*x*y-2*z*w, 2*x*z+2*y*w],
        [2*x*y+2*z*w, 1-2*x**2-2*z**2, 2*y*z-2*x*w],
        [2*x*z-2*y*w, 2*y*z+2*x*w, 1-2*x**2-2*y**2]
    ])

colmap_dir = os.path.join(scene_dir, 'dslr/colmap')
image_dir = os.path.join(scene_dir, 'dslr/resized_images')
object_dir = scene_dir + '/extracted_individual_objects'
output_crop_dir = scene_dir + '/object_crops'
os.makedirs(output_crop_dir, exist_ok=True)

In [8]:
'''
for obj_file in os.listdir(object_dir):
    if not obj_file.endswith('.ply'):
        continue
    obj_path = os.path.join(object_dir, obj_file)
    obj_mesh = o3d.io.read_triangle_mesh(obj_path)
    vertices = np.asarray(obj_mesh.vertices)
    print(f"Processing object: {obj_file}")

    # Loop over all images
    for img_name, img_info in images.items():
        img_path = os.path.join(image_dir, img_name)
        if not os.path.exists(img_path):
            continue

        qvec, tvec, cam_id = img_info['qvec'], img_info['tvec'], img_info['camera_id']
        cam = cameras[cam_id]
        R = qvec2rotmat(qvec)
        t = tvec.reshape((3,1))
        K = np.array([[cam['fx'], 0, cam['cx']],
                      [0, cam['fy'], cam['cy']],
                      [0, 0, 1]])
        w, h = cam['width'], cam['height']

        # Transform vertices
        verts_cam = (R @ vertices.T).T + t.T
        proj = (K @ verts_cam.T).T
        u, v, z = proj[:, 0] / proj[:, 2], proj[:, 1] / proj[:, 2], proj[:, 2]
        valid = (z > 0) & (u >= 0) & (u < w) & (v >= 0) & (v < h)
        if np.sum(valid) == 0:
            continue  # Not visible

        min_u, max_u = int(np.min(u[valid])), int(np.max(u[valid]))
        min_v, max_v = int(np.min(v[valid])), int(np.max(v[valid]))

        # Ensure bounding box is valid
        # Extend the 2D Mask Bounding Boxes to preserve information
        padding = 20
        min_u = max(0, min_u - padding)
        max_u = min(w, max_u + padding)
        min_v = max(0, min_v - padding)
        max_v = min(h, max_v + padding)
        if max_u > min_u and max_v > min_v:
            area = (max_u - min_u) * (max_v - min_v)
            proj_areas.append((area, img_name, min_u, max_u, min_v, max_v))
        if max_u > min_u and max_v > min_v:
            image = np.array(Image.open(img_path))
            crop = image[min_v:max_v, min_u:max_u]
            crop_name = f"{os.path.splitext(img_name)[0]}_{os.path.splitext(obj_file)[0]}.jpg"
            cv2.imwrite(os.path.join(output_crop_dir, crop_name), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
            print(f"Saved crop: {crop_name}")
        else:
            print(f"Skipping crop for {img_name} (empty or invalid bounding box)")
'''

'\nfor obj_file in os.listdir(object_dir):\n    if not obj_file.endswith(\'.ply\'):\n        continue\n    obj_path = os.path.join(object_dir, obj_file)\n    obj_mesh = o3d.io.read_triangle_mesh(obj_path)\n    vertices = np.asarray(obj_mesh.vertices)\n    print(f"Processing object: {obj_file}")\n\n    # Loop over all images\n    for img_name, img_info in images.items():\n        img_path = os.path.join(image_dir, img_name)\n        if not os.path.exists(img_path):\n            continue\n\n        qvec, tvec, cam_id = img_info[\'qvec\'], img_info[\'tvec\'], img_info[\'camera_id\']\n        cam = cameras[cam_id]\n        R = qvec2rotmat(qvec)\n        t = tvec.reshape((3,1))\n        K = np.array([[cam[\'fx\'], 0, cam[\'cx\']],\n                      [0, cam[\'fy\'], cam[\'cy\']],\n                      [0, 0, 1]])\n        w, h = cam[\'width\'], cam[\'height\']\n\n        # Transform vertices\n        verts_cam = (R @ vertices.T).T + t.T\n        proj = (K @ verts_cam.T).T\n        u, v

In [9]:
from collections import defaultdict
import random
cameras = read_cameras_text(os.path.join(colmap_dir, 'cameras.txt'))
images = read_images_text(os.path.join(colmap_dir, 'images.txt'))

output_crop_dir = os.path.join(scene_dir, 'object_crops')
os.makedirs(output_crop_dir, exist_ok=True)
output_top5_dir = os.path.join(scene_dir, 'cropped_top5')
os.makedirs(output_top5_dir, exist_ok=True)

# Save the projection area of all images for each obj
object_areas = defaultdict(list)

# Loop through all extracted objects
for obj_file in os.listdir(object_dir):
    if not obj_file.endswith('.ply'):
        continue
    obj_path = os.path.join(object_dir, obj_file)
    obj_mesh = o3d.io.read_triangle_mesh(obj_path)
    vertices = np.asarray(obj_mesh.vertices)
    print(f"Processing object: {obj_file}")

    for img_name, img_info in images.items():
        img_path = os.path.join(image_dir, img_name)
        if not os.path.exists(img_path):
            continue

        qvec, tvec, cam_id = img_info['qvec'], img_info['tvec'], img_info['camera_id']
        cam = cameras[cam_id]
        R = qvec2rotmat(qvec)
        t = np.array(tvec).reshape((3,1))
        K = np.array([[cam['fx'], 0, cam['cx']],
                      [0, cam['fy'], cam['cy']],
                      [0, 0, 1]])
        w, h = cam['width'], cam['height']

        # Make the projection from 3D to 2D
        verts_cam = (R @ vertices.T).T + t.T
        proj = (K @ verts_cam.T).T
        u, v, z = proj[:, 0] / proj[:, 2], proj[:, 1] / proj[:, 2], proj[:, 2]
        valid = (z > 0) & (u >= 0) & (u < w) & (v >= 0) & (v < h)
        if np.sum(valid) == 0:
            continue

        min_u, max_u = int(np.min(u[valid])), int(np.max(u[valid]))
        min_v, max_v = int(np.min(v[valid])), int(np.max(v[valid]))

        # padding
        padding = 0
        min_u = max(0, min_u - padding)
        max_u = min(w, max_u + padding)
        min_v = max(0, min_v - padding)
        max_v = min(h, max_v + padding)

        if max_u > min_u and max_v > min_v:
            area = (max_u - min_u) * (max_v - min_v)
            object_areas[obj_file].append((area, img_name, min_u, max_u, min_v, max_v))

            # Save all cropped images
            image = np.array(Image.open(img_path))
            crop = image[min_v:max_v, min_u:max_u]
            crop_name = f"{os.path.splitext(obj_file)[0]}_{os.path.splitext(img_name)[0]}.jpg"
            crop_path = os.path.join(output_crop_dir, crop_name)
            cv2.imwrite(crop_path, cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
        else:
            print(f"Skipping invalid crop for {img_name}")

for obj_file, views in object_areas.items():
    # extract Top-3 projections
    views_sorted = sorted(views, key=lambda x: x[0], reverse=True)
    top_5_view = views_sorted[:3]
    for area, img_name, min_u, max_u, min_v, max_v in top_5_view:
        img_path = os.path.join(image_dir, img_name)
        image = np.array(Image.open(img_path))
        crop = image[min_v:max_v, min_u:max_u]
        crop_name = f"{os.path.splitext(obj_file)[0]}_{os.path.splitext(img_name)[0]}.jpg"
        crop_path = os.path.join(output_top5_dir, crop_name)
        cv2.imwrite(crop_path, cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
        print(f"Saved top5 crop: {crop_name}")

    # extract Random 2 projections
    remaining_views = views_sorted[3:]

    if len(remaining_views) >= 3:
        random_views = random.sample(remaining_views, 2)
    else:
        random_views = remaining_views

    for area, img_name, min_u, max_u, min_v, max_v in random_views:
        img_path = os.path.join(image_dir, img_name)
        image = np.array(Image.open(img_path))
        crop = image[min_v:max_v, min_u:max_u]
        crop_name = f"{os.path.splitext(obj_file)[0]}_{os.path.splitext(img_name)[0]}_random.jpg"
        crop_path = os.path.join(output_top5_dir, crop_name)
        cv2.imwrite(crop_path, cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
        print(f"Saved random crop: {crop_name}")

Processing object: object_65_pipe.ply
Processing object: object_15_storage cabinet.ply
Skipping invalid crop for DSC03572.JPG
Processing object: object_29_door.ply
Processing object: object_107_SPLIT.ply
Skipping invalid crop for DSC03649.JPG
Skipping invalid crop for DSC03645.JPG
Skipping invalid crop for DSC03644.JPG
Processing object: object_63_SPLIT.ply
Skipping invalid crop for DSC03617.JPG
Processing object: object_5_ceiling lamp.ply
Processing object: object_3_wall.ply
Processing object: object_19_bin.ply
Processing object: object_95_SPLIT.ply
Processing object: object_39_monitor.ply
Processing object: object_97_binder.ply
Processing object: object_136_ceiling light.ply
Processing object: object_137_carboard box.ply
Processing object: object_99_cardboard box.ply
Skipping invalid crop for DSC03464.JPG
Processing object: object_28_laptop.ply
Skipping invalid crop for DSC03581.JPG
Processing object: object_74_object.ply
Processing object: object_129_ceiling light.ply
Processing obj

Processing object: object_69_power socket.ply
Processing object: object_32_whiteboard marker.ply
Processing object: object_114_laptop stand.ply
Skipping invalid crop for DSC03581.JPG
Processing object: object_105_object.ply
Processing object: object_124_window sill.ply
Processing object: object_89_stack of paper.ply
Skipping invalid crop for DSC03500.JPG
Processing object: object_37_power socket.ply
Processing object: object_8_ceiling lamp.ply
Processing object: object_43_wall.ply
Skipping invalid crop for DSC03620.JPG
Processing object: object_6_ceiling lamp.ply
Skipping invalid crop for DSC03631.JPG
Processing object: object_36_light switch.ply
Processing object: object_91_paper.ply
Processing object: object_60_REMOVE.ply
Processing object: object_21_office chair.ply
Processing object: object_1_whiteboard.ply
Processing object: object_77_keyboard.ply
Skipping invalid crop for DSC03469.JPG
Processing object: object_120_REMOVE.ply
Processing object: object_78_box.ply
Processing object:

In [10]:
folder_path = f'{scene_dir}/object_crops'
file_list = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
print(f'Number of cropped images: {len(file_list)}')

Number of cropped images: 11689


In [11]:
folder_path = f'{scene_dir}/object_crops'
obj = 1
file_list = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f)) and f'object_{obj}_' in f]
print(f'Number of object {obj}: {len(file_list)}')

Number of object 1: 91


In [12]:
folder_path = f'{scene_dir}/dslr/resized_images'
file_list = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
print(f'Number of multi-view images: {len(file_list)}')

Number of multi-view images: 232


In [13]:
folder_path = f'{scene_dir}/cropped_top5'
file_list = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
print(f'Number of cropped multi-view images: {len(file_list)}')

Number of cropped multi-view images: 957


# 4. Use SOTA VLM to generate object-level captions.

In [1]:
from lmdeploy import pipeline, TurbomindEngineConfig, ChatTemplateConfig
from lmdeploy.vl.constants import IMAGE_TOKEN

model = 'OpenGVLab/InternVL3-2B'
pipe = pipeline(model, backend_config=TurbomindEngineConfig(session_len=16384, tp=1), chat_template_config=ChatTemplateConfig(model_name='internvl2_5'))

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

ImportError: cannot import name 'ImagesKwargs' from 'transformers.processing_utils' (/home/vlm_caption/miniconda3/envs/maskclustering/lib/python3.9/site-packages/transformers/processing_utils.py)

In [ ]:
# Numbering images improves multi-image conversations
question = ''.join([f'Image-{i+1}: {IMAGE_TOKEN}\n' for i in range(5)])
question += 'You are provided with top-5 views of one individual object selected based on the projection area. Please describe this individual object in details, including its appearance, potential function, and properties.'

In [ ]:
folder_path = f'{scene_dir}/cropped_top5'
file_list = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

"\nimage_paths = [f'{scene_dir}/cropped_top5/{img}' for img in file_list]\nimages = [Image.open(path) for path in image_paths][:7]\n"

In [ ]:
objects_images = []
for i in range(len(unique_instances)):
    folder_path = f'{scene_dir}/cropped_top5'
    image_path = [f'{scene_dir}/cropped_top5/{path}' for path in file_list if f'object_{i+1}_' in path]
    filename = os.path.basename(image_path[0])
    parts = filename.split('_')
    object_name = parts[2]
    objects_images.append(
        {
            f'{i+1}_{object_name}': image_path[:6]
        }
    )
print(objects_images)

[{'1_whiteboard': ['ScanNet_Data/data/56a0ec536c/cropped_top5/object_1_whiteboard_DSC03629.jpg', 'ScanNet_Data/data/56a0ec536c/cropped_top5/object_1_whiteboard_DSC03628.jpg', 'ScanNet_Data/data/56a0ec536c/cropped_top5/object_1_whiteboard_DSC03683_random.jpg', 'ScanNet_Data/data/56a0ec536c/cropped_top5/object_1_whiteboard_DSC03703.jpg', 'ScanNet_Data/data/56a0ec536c/cropped_top5/object_1_whiteboard_DSC03512_random.jpg']}, {'2_blinds': ['ScanNet_Data/data/56a0ec536c/cropped_top5/object_2_blinds_DSC03690_random.jpg', 'ScanNet_Data/data/56a0ec536c/cropped_top5/object_2_blinds_DSC03661.jpg', 'ScanNet_Data/data/56a0ec536c/cropped_top5/object_2_blinds_DSC03711_random.jpg', 'ScanNet_Data/data/56a0ec536c/cropped_top5/object_2_blinds_DSC03659.jpg', 'ScanNet_Data/data/56a0ec536c/cropped_top5/object_2_blinds_DSC03658.jpg']}, {'3_wall': ['ScanNet_Data/data/56a0ec536c/cropped_top5/object_3_wall_DSC03453.jpg', 'ScanNet_Data/data/56a0ec536c/cropped_top5/object_3_wall_DSC03705_random.jpg', 'ScanNet_Dat

In [ ]:
objects_captions = []
for i in range(len(objects_images)):
    label, image_paths = next(iter(objects_images[i].items()))
    images = [Image.open(path) for path in image_paths]
    response = pipe((question, images))
    objects_captions.append({label:response.text})

2025-06-04 21:43:51,267 - lmdeploy - ERROR - async_engine.py:692 - Truncate max_new_tokens to 128
2025-06-04 21:43:51,267 - lmdeploy - ERROR - async_engine.py:694 - run out of tokens. session=215.


In [ ]:
objects_captions[10].items()

dict_items([('11_whiteboard', "The image depicts a whiteboard mounted on a wall, with a curved design that suggests it might be a flexible or retractable panel. The whiteboard is blank, indicating it's ready for use. There are a few markers attached to the bottom of the board, suggesting it's used for writing or drawing. The room appears to be a meeting or conference room, as indicated by the presence of chairs and the setup. The lighting is bright, and the overall color scheme is neutral, with the whiteboard being the most prominent feature. The curved design of the whiteboard could be functional, allowing for easier access to the writing surface from different angles.")])

In [17]:
import json
with open('object_level_captions.json', 'w') as f:
    json.dump(objects_captions, f, indent=2)

# Check if the generated caption indeed described the corresponding objects


In [2]:
import json
with open('object_level_captions.json', 'r') as f:
    object_captions = json.load(f)
print(object_captions)

[{'1_whiteboard': "The image displays a whiteboard, a common tool used in educational and professional settings for brainstorming, note-taking, and presentations. The whiteboard is mounted on a wall and appears to be in a room with a carpeted floor. It is covered with various drawings and writings, suggesting it has been used for brainstorming or planning. The drawings include shapes, lines, and possibly diagrams, indicating a creative or analytical process. The whiteboard's surface is smooth and reflective, typical of such boards, and it is supported by a metal frame. The room's lighting is bright, illuminating the whiteboard and its contents. The whiteboard's properties include its ability to hold multiple notes and drawings, its durability, and its versatility for different types of writing and drawing."}, {'2_blinds': "The image showcases a vertical, beige-colored panel with a ribbed texture, resembling a curtain or a panel divider. It is attached to a curved wall, suggesting it mi

In [3]:
import evaluate
# pip install bert_score
bertscore = evaluate.load('bertscore')

In [5]:
predictions = []
references = []
for i in range(len(object_captions)):
    label, caption = next(iter(object_captions[i].items()))
    predictions.append(caption)
    references.append(label)

similarity_scores = bertscore.compute(predictions=predictions, references=references, lang = 'en')
'''
for i in range(len(references)):
    print(f"Label: {references[i]}")
    print(f"Caption: {predictions[i]}")
    print(f"BERTScore F1: {similarity_scores['f1'][i]:.4f}\n")
'''

'\nfor i in range(len(references)):\n    print(f"Label: {references[i]}")\n    print(f"Caption: {predictions[i]}")\n    print(f"BERTScore F1: {similarity_scores[\'f1\'][i]:.4f}\n")\n'

In [17]:
f1_scores = similarity_scores.get('f1')
num_zeros = sum(1 for score in f1_scores if score == 0)
sum_bertscore = sum(score for score in f1_scores if score != 0)
print(f"Number of zero F1 scores: {num_zeros}")
print(f"Mean BERTScore F1: {sum_bertscore/(138-num_zeros)}")

Number of zero F1 scores: 1
Mean BERTScore F1: 0.789192302818716


In [18]:
import numpy as np
mean_f1 = np.mean(similarity_scores.get('f1'))
print(f"Mean BERTScore F1: {mean_f1:.4f}")

Mean BERTScore F1: 0.7835


In [27]:
number_low = 0
f1_scores = similarity_scores['f1']
for score in f1_scores:
    if score < 0.7 and score > 0:
        number_low += 1
print(number_low)

0


# Check Cuda Availability

In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
print("Current device:", torch.cuda.current_device())
print("Device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

In [ ]:
print("Memory Allocated:", torch.cuda.memory_allocated() / 1024**2, "MB")
print("Memory Reserved:", torch.cuda.memory_reserved() / 1024**2, "MB")
print("Max Memory Allocated:", torch.cuda.max_memory_allocated() / 1024**2, "MB")
print("Max Memory Reserved:", torch.cuda.max_memory_reserved() / 1024**2, "MB")

In [ ]:
!nvidia-smi

Wed Jun  4 13:10:29 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.51.03              Driver Version: 575.51.03      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3080        Off |   00000000:01:00.0 Off |                  N/A |
| 30%   47C    P8             17W /  320W |      38MiB /  10240MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import math
import numpy as np
import torch
import torchvision.transforms as T
from decord import VideoReader, cpu
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer, AutoConfig

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height

    # calculate the existing image aspect ratio
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if
        i * j <= max_num and i * j >= min_num)
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])

    # find the closest aspect ratio to the target
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)

    # calculate the target width and height
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]

    # resize the image
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        # split the image
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_file, input_size=448, max_num=12):
    image = Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values

def split_model(model_name):
    device_map = {}
    world_size = torch.cuda.device_count()
    model_path = 'OpenGVLab/InternVL3-2B'
    config = AutoConfig.from_pretrained(model_path, trust_remote_code=True)
    num_layers = config.llm_config.num_hidden_layers
    # Since the first GPU will be used for ViT, treat it as half a GPU.
    num_layers_per_gpu = math.ceil(num_layers / (world_size - 0.5))
    num_layers_per_gpu = [num_layers_per_gpu] * world_size
    num_layers_per_gpu[0] = math.ceil(num_layers_per_gpu[0] * 0.5)
    layer_cnt = 0
    for i, num_layer in enumerate(num_layers_per_gpu):
        for j in range(num_layer):
            device_map[f'language_model.model.layers.{layer_cnt}'] = i
            layer_cnt += 1
    device_map['vision_model'] = 0
    device_map['mlp1'] = 0
    device_map['language_model.model.tok_embeddings'] = 0
    device_map['language_model.model.embed_tokens'] = 0
    device_map['language_model.output'] = 0
    device_map['language_model.model.norm'] = 0
    device_map['language_model.model.rotary_emb'] = 0
    device_map['language_model.lm_head'] = 0
    device_map[f'language_model.model.layers.{num_layers - 1}'] = 0

    return device_map

In [ ]:
path = 'OpenGVLab/InternVL3-2B'
device_map = split_model('InternVL3-2B')
model = AutoModel.from_pretrained(
    path,
    torch_dtype=torch.bfloat16,
    load_in_8bit=False,
    low_cpu_mem_usage=True,
    use_flash_attn=True,
    trust_remote_code=True,
    device_map=device_map).eval()
tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True, use_fast=False)
generation_config = dict(max_new_tokens=1024, do_sample=True)

In [ ]:
# multi-image multi-round conversation, separate images.
file_list = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f)) and 'object_1_' in f]
image_paths = [f'{scene_dir}/cropped_top5/{img}' for img in file_list]

pixel_value_list = []
num_patches_list = []
for path in image_paths:
    pixel_values = load_image(path, max_num=12).to(torch.bfloat16).cuda()
    pixel_value_list.append(pixel_values)
    num_patches_list.append(pixel_values.shape[0])  # how many patches this image contributes

pixel_values = torch.cat(pixel_value_list, dim=0)  # shape: (total_patches, 3, H, W)

question = ''.join([f'Image-{i+1}: <image>\n' for i in range(len(image_paths))])
question += 'We are working on a 3D scene captioning task. The scene has been decomposed into individual objects, and for each object, you are provided with its top-5 views selected based on the projection area. Please describe individual object in detail, including its appearance, potential function.'

# question = 'Image-1: <image>\nImage-2: <image>\nDescribe these images in detail.'

response, history = model.chat(tokenizer, pixel_values, question, generation_config,
                               num_patches_list=num_patches_list,
                               history=None, return_history=True)

print(f'User: {question}\nAssistant: {response}')

In [ ]:
# !pip install lmdeploy
# !pip install timm
# !pip install flash-attn --no-build-isolation

In [2]:
import transformers
print(transformers.__version__)
print(transformers.__file__)

4.52.4
/home/vlm_caption/miniconda3/envs/maskclustering/lib/python3.9/site-packages/transformers/__init__.py
